In [2]:
import copy
import h5py
import numpy as np
import pandas as pd
from collections import defaultdict
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

In [3]:
INIT_TRAIN_RATIO    = 0.80
REVIEW_BUDGET       = 0.20   # max fraction of block sent to human review
CDF_OUTLIER         = 0.80   # CDF > this  → review queue
CDF_REDUNDANT       = 0.20   # CDF < this  → discard, no update
NEW_SPECIES_MIN_N   = 20     # confirmed samples before p95 is trusted for new species
INLIER_WINDOW       = 2000    # max inlier distances kept per sub-prototype

H5_PATH = "../embeddings/cct_dinov2l_embeddings_v2.h5"
N_BLOCKS = 5          # blocks PER GROUP (5 city + 5 countryside = 10 total)

print("Configuration set.")
print(f"  H5 path        : {H5_PATH}")
print(f"  Blocks per group: {N_BLOCKS}")


Configuration set.
  H5 path        : ../embeddings/cct_dinov2l_embeddings_v2.h5
  Blocks per group: 5


In [4]:
EXCLUDED_LOCATIONS = {"71", "118", "47", "80", "135", "61", "85", "41"}

# ── 1. Load & clean ────────────────────────────────────────────────────────────
with h5py.File(H5_PATH, "r") as hf:
    raw_embeddings = hf["embeddings"][:]
    raw_species    = np.array([s.decode() for s in hf["species"][:]])
    raw_strings    = [t.decode() for t in hf["date_captured"][:]]
    raw_locations  = np.array([l.decode() for l in hf["location"][:]])

temp_times = pd.to_datetime(raw_strings, errors="coerce")

# Combined mask: valid timestamp AND not in excluded locations
location_mask = ~np.isin(raw_locations, list(EXCLUDED_LOCATIONS))
mask          = temp_times.notna() & location_mask

embeddings = normalize(raw_embeddings[mask], norm="l2")
species    = raw_species[mask]
timestamps = temp_times[mask]
locations  = raw_locations[mask]

print(f"Loaded {len(embeddings):,} embeddings after cleaning.")
print(f"Dropped {(~temp_times.notna()).sum()} rows with invalid timestamps.")
print(f"Dropped {(~location_mask).sum()} rows from excluded locations.")
print(f"Dropped {(~mask).sum()} rows total.")

Loaded 103,095 embeddings after cleaning.
Dropped 1 rows with invalid timestamps.
Dropped 4054 rows from excluded locations.
Dropped 4055 rows total.


In [5]:
df = pd.DataFrame({"location": locations, "timestamp": timestamps})

rough_threshold = pd.Timestamp("2013-07-01")

location_counts = (
    df.groupby("location")
    .apply(lambda x: pd.Series({
        "count_early": (x["timestamp"] < rough_threshold).sum(),
        "count_late":  (x["timestamp"] >= rough_threshold).sum(),
    }))
    .reset_index()
)
location_counts["assigned_group"] = np.where(
    location_counts["count_early"] >= location_counts["count_late"],
    "group1", "group2"
)

group1_locs = set(location_counts[location_counts["assigned_group"] == "group1"]["location"])
group2_locs = set(location_counts[location_counts["assigned_group"] == "group2"]["location"])
assert len(group1_locs & group2_locs) == 0, "Location overlap between groups!"

borderline = location_counts[
    (location_counts["count_early"] > 0) & (location_counts["count_late"] > 0)
]
print(f"\nBorderline locations (data on both sides of threshold): {len(borderline)}")
print(borderline[["location", "count_early", "count_late", "assigned_group"]].to_string())
print(f"\nGroup 1 locations: {len(group1_locs)}")
print(f"Group 2 locations: {len(group2_locs)}")


Borderline locations (data on both sides of threshold): 0
Empty DataFrame
Columns: [location, count_early, count_late, assigned_group]
Index: []

Group 1 locations: 41
Group 2 locations: 91


In [6]:
# ── 3. Sort ALL data chronologically, then derive per-group index arrays ───────
sort_idx   = np.argsort(timestamps)
embeddings = embeddings[sort_idx]
species    = species[sort_idx]
timestamps = timestamps[sort_idx]
locations  = locations[sort_idx]

dates_arr = np.array([d.date() for d in timestamps])

# Boolean masks in the sorted array
g1_mask = np.isin(locations, list(group1_locs))
g2_mask = np.isin(locations, list(group2_locs))

# Absolute indices (into the sorted arrays) for each group
g1_indices = np.where(g1_mask)[0]
g2_indices = np.where(g2_mask)[0]

print(f"\nData sorted chronologically: {timestamps.min()} → {timestamps.max()}")
print(f"  Group 1 samples : {len(g1_indices):,}")
print(f"  Group 2 samples : {len(g2_indices):,}")
print(f"  Total           : {len(embeddings):,}")


Data sorted chronologically: 2010-05-25 20:32:56 → 2015-06-03 12:07:41
  Group 1 samples : 57,359
  Group 2 samples : 45,736
  Total           : 103,095


In [7]:
# ── 4. Date-aware block partitioning ──────────────────────────────────────────
def make_date_aware_blocks(dates_arr, global_indices, n_blocks):
    """
    Partition `global_indices` into `n_blocks` roughly-equal temporal blocks
    so that no calendar date is split across two blocks.

    Parameters
    ----------
    dates_arr     : np.ndarray of datetime.date, shape (N_total,)
                    Date for every sample in the globally-sorted array.
    global_indices: np.ndarray of int
                    Absolute positions (into dates_arr / embeddings / …) that
                    belong to this group, already in ascending time order.
    n_blocks      : int

    Returns
    -------
    List of np.ndarray – each array contains absolute indices for one block.
    """
    N = len(global_indices)
    target_block_size = N / n_blocks

    # Map each date to the global indices that fall on it (within this group)
    date_to_indices = defaultdict(list)
    for gi in global_indices:
        date_to_indices[dates_arr[gi]].append(gi)

    unique_dates = sorted(date_to_indices.keys())
    blocks    = [[] for _ in range(n_blocks)]
    cur_block = 0
    cumulative = 0

    for d in unique_dates:
        idxs = date_to_indices[d]
        blocks[cur_block].extend(idxs)
        cumulative += len(idxs)
        if cur_block < n_blocks - 1 and cumulative >= (cur_block + 1) * target_block_size:
            cur_block += 1

    return [np.array(b) for b in blocks if len(b) > 0]


g1_blocks = make_date_aware_blocks(dates_arr, g1_indices, N_BLOCKS)
g2_blocks = make_date_aware_blocks(dates_arr, g2_indices, N_BLOCKS)

In [8]:
# ── 5. Summary ────────────────────────────────────────────────────────────────
def print_blocks(label, blocks, dates_arr):
    print(f"\n{'─'*60}")
    print(f"  {label}  ({len(blocks)} blocks)")
    print(f"  {'Block':>6}  {'Size':>7}  {'Date range'}")
    print(f"  {'─'*52}")
    for i, b in enumerate(blocks):
        d_min = dates_arr[b].min()
        d_max = dates_arr[b].max()
        print(f"    {i+1:>4}  {len(b):>7,}  {d_min}  →  {d_max}")

print_blocks("Group 1 (city)", g1_blocks, dates_arr)
print_blocks("Group 2 (countryside)", g2_blocks, dates_arr)

total = sum(len(b) for b in g1_blocks) + sum(len(b) for b in g2_blocks)
print(f"\nTotal samples across all blocks: {total:,}  (expected {len(embeddings):,})")
assert total == len(embeddings), "Sample count mismatch – some samples lost!"


────────────────────────────────────────────────────────────
  Group 1 (city)  (5 blocks)
   Block     Size  Date range
  ────────────────────────────────────────────────────
       1   11,642  2010-05-25  →  2011-06-19
       2   11,339  2011-06-20  →  2011-10-05
       3   11,495  2011-10-06  →  2012-01-16
       4   11,537  2012-01-17  →  2012-04-12
       5   11,346  2012-04-13  →  2013-05-10

────────────────────────────────────────────────────────────
  Group 2 (countryside)  (5 blocks)
   Block     Size  Date range
  ────────────────────────────────────────────────────
       1    9,233  2013-08-02  →  2013-11-27
       2    9,092  2013-11-28  →  2014-02-06
       3    9,124  2014-02-07  →  2014-05-08
       4    9,182  2014-05-09  →  2014-09-13
       5    9,105  2014-09-14  →  2015-06-03

Total samples across all blocks: 103,095  (expected 103,095)


In [9]:
# ══════════════════════════════════════════════════════════════════════
# 1.  GALLERY-BASED CLASSIFIER
# ══════════════════════════════════════════════════════════════════════
class GalleryNCMClassifier:
    """
    Nearest-centroid classifier backed by a structured gallery.

    gallery[species] = list of sub-prototype dicts:
        {
          "proto"       : np.ndarray  (L2-normalised, shape D)
          "threshold"   : float       (p95 cosine distance)
          "label"       : str         (e.g. "deer" or "deer_c1")
          "n"           : int         (confirmed samples contributing)
          "inlier_dists": np.ndarray  (sorted, last INLIER_WINDOW values)
        }
    """

    def __init__(self):
        self.gallery: dict[str, list[dict]] = {}

    # ── construction ────────────────────────────────────────────────
    def fit(self, embs: np.ndarray, lbls: np.ndarray,
            subprototype_counts = None):
        """
        Initialise gallery from support embeddings.
        subprototype_counts: {species: k}  – defaults to 1 for missing keys.
        embs must already be L2-normalised.
        """
        if subprototype_counts is None:
            subprototype_counts = {}

        self.gallery = {}
        
        for sp in np.unique(lbls):
            k    = subprototype_counts.get(sp, 1)
            mask = (lbls == sp)
            self._init_species(sp, embs[mask], k)

    def _init_species(self, sp: str, embs: np.ndarray, k: int):
        """Cluster embs into k sub-prototypes and store gallery entries."""
        k = min(k, len(embs))
        
        if k == 1:
            assignments = np.zeros(len(embs), dtype=int)
        else:
            km          = KMeans(n_clusters=k, n_init=10, random_state=42)
            assignments = km.fit_predict(embs)
        
        self.gallery[sp] = []
        
        for c in range(k):
            mask_c = (assignments == c)
            
            if mask_c.sum() < 1:
                continue
            
            sub   = embs[mask_c]
            n     = int(mask_c.sum())

            # un-normalized running sum — used for exact prototype recomputation
            emb_sum = sub.sum(axis=0)
            
            # prototype = normalized mean
            proto = emb_sum / n
            proto = proto / np.linalg.norm(proto)
            
            dists = 1.0 - (sub @ proto).clip(-1, 1)
            
            self.gallery[sp].append({
                "proto"       : proto,
                "emb_sum"     : emb_sum,          # accumulated for future updates
                "threshold"   : float(np.percentile(dists, 95)),
                "label"       : sp if k == 1 else f"{sp}_c{c}",
                "n"           : n,
                "inlier_dists": np.sort(dists),
            })

    # ── inference ───────────────────────────────────────────────────
    def _nearest(self, emb: np.ndarray):
        """Return (species, entry, cosine_distance) for nearest sub-prototype."""
        best_sp, best_entry, best_dist = None, None, np.inf
        
        for sp, entries in self.gallery.items():
            for entry in entries:
                d = 1.0 - float(np.dot(emb, entry["proto"]).clip(-1, 1))
                if d < best_dist:
                    best_sp, best_entry, best_dist = sp, entry, d
        
        return best_sp, best_entry, best_dist

    def predict(self, emb: np.ndarray) -> str:
        sp, _, _ = self._nearest(emb)
        return sp

    def cdf_score(self, emb: np.ndarray):
        """
        Returns (predicted_species, cdf_score, distance, accepted).
        accepted = distance <= p95 threshold of nearest sub-prototype.
        """
        sp, entry, dist = self._nearest(emb)
        ids = entry["inlier_dists"]
        cdf = float(np.searchsorted(ids, dist, side="right") / len(ids)) \
              if len(ids) > 0 else 0.5
       
        accepted = (dist <= entry["threshold"])
        
        return sp, cdf, dist, accepted

    # ── gallery update (end-of-block) ───────────────────────────────
    def recompute_from_confirmed(self, confirmed):
        """
        Recompute prototype vectors and p95 thresholds from all confirmed
        inlier samples collected during the block.

        confirmed: list of {"embedding": ndarray, "confirmed_species": str}
        """
        # group by species
        by_sp: dict[str, list] = defaultdict(list)
        for s in confirmed:
            by_sp[s["confirmed_species"]].append(s["embedding"])

        for sp, emb_list in by_sp.items():
            embs = np.array(emb_list)

            if sp not in self.gallery:
                # new species – bootstrap with k=1
                self._init_species(sp, embs, k=1)
                # if too few samples, keep conservative threshold
                if len(embs) < NEW_SPECIES_MIN_N:
                    med = np.median([e["threshold"]
                                     for s_ in self.gallery
                                     for e in self.gallery[s_]
                                     if s_ != sp])
                    self.gallery[sp][0]["threshold"] = float(med)
                continue

            # assign each new embedding to its nearest existing sub-prototype
            entries     = self.gallery[sp]
            proto_mat   = np.stack([e["proto"] for e in entries])   # (K, D)
            assignments = (embs @ proto_mat.T).argmax(axis=1)       # (N,)

            for c, entry in enumerate(entries):
                mask_c = (assignments == c)
                
                if mask_c.sum() == 0:
                    continue
    
                sub = embs[mask_c]

                # accumulate into running sum
                entry["emb_sum"] += sub.sum(axis=0)
                entry["n"]       += int(mask_c.sum())

                # recompute prototype from full history
                new_proto = entry["emb_sum"] / entry["n"]
                entry["proto"] = new_proto / np.linalg.norm(new_proto)

                # accumulate inlier distances and recompute threshold
                new_dists = 1.0 - (sub @ entry["proto"]).clip(-1, 1)
                all_dists = np.sort(np.concatenate([entry["inlier_dists"], new_dists]))[-INLIER_WINDOW:]
                entry["inlier_dists"] = all_dists
                entry["threshold"]    = float(np.percentile(all_dists, 95))

    @property
    def known_species(self):
        return set(self.gallery.keys())

In [12]:
# ══════════════════════════════════════════════════════════════════════
# 2.  BLOCK PROCESSING WITH CDF-BASED REVIEW
# ══════════════════════════════════════════════════════════════════════
def process_block(clf: GalleryNCMClassifier,
                  blk_embs: np.ndarray,
                  blk_lbls: np.ndarray,
                  review_budget: float = REVIEW_BUDGET) -> dict:
    """
    Process one block through the full pipeline:

      1. Score every sample (predict + CDF)
      2. Flag CDF > 0.95 samples; cap at review_budget
      3. Simulate human review using ground-truth labels
      4. Auto-update on informative inliers (0.50 <= CDF <= 0.95
         AND distance <= p95 threshold)
      5. Recompute gallery statistics from confirmed samples
      6. Compute accuracy metrics (prediction-only, all samples)

    Returns a metrics dict compatible with your existing print_results().
    """
    N         = len(blk_embs)
    n_review  = max(1, int(N * review_budget))
    
    known_species_prev = clf.known_species

    # ── step 1: score every sample ───────────────────────────────────
    records = []
    for i, (emb, lbl) in enumerate(zip(blk_embs, blk_lbls)):
        sp, cdf, dist, accepted = clf.cdf_score(emb)
        records.append({
            "idx"     : i,
            "emb"     : emb,
            "gt"      : lbl,
            "pred"    : sp,
            "cdf"     : cdf,
            "dist"    : dist,
            "accepted": accepted,
            "correct" : sp == lbl,
        })

    # ── step 2: review queue – CDF > 0.95, capped at budget ──────────
    outliers = [r for r in records if r["cdf"] > CDF_OUTLIER]
    outliers.sort(key=lambda r: r["cdf"], reverse=True)
    #review_queue = outliers[:n_review]
    review_queue = outliers
    review_set   = {r["idx"] for r in review_queue}
    
    n_actual_new_species_images = sum(
        1 for r in records if r["gt"] not in known_species_prev
    )
    actual_new_species = set(
        r["gt"] for r in records if r["gt"] not in known_species_prev
    )
    
    # of those, how many ended up in review queue (checked after the fact)
    n_new_species_in_queue = sum(
        1 for r in review_queue if r["gt"] not in known_species_prev
    )
    new_species_recall = (
        n_new_species_in_queue / n_actual_new_species_images
        if n_actual_new_species_images > 0 else float("nan")
    )

    # ── step 3: simulate human review with ground truth ───────────────
    confirmed = []
    for r in review_queue:
        confirmed.append({
            "embedding"        : r["emb"],
            "confirmed_species": r["gt"],          # ground truth = human label
        })

    # ── step 4: auto-update on informative inliers ────────────────────
    # conditions: not in review queue
    #             AND distance <= p95 threshold   (accepted)
    #             AND 0.50 <= CDF <= 0.95         (informative)
    for r in records:
        if r["idx"] in review_set:
            continue
        if r["accepted"] and CDF_REDUNDANT <= r["cdf"] <= CDF_OUTLIER:
            confirmed.append({
                "embedding"        : r["emb"],
                "confirmed_species": r["gt"],
            })

    # ── step 5: recompute gallery from all confirmed samples ──────────
    clf.recompute_from_confirmed(confirmed)

    # ── step 6: accuracy metrics (prediction-only, all samples) ───────
    species_in_block = sorted(set(blk_lbls))
    sp_correct = {sp: 0 for sp in species_in_block}
    sp_total   = {sp: 0 for sp in species_in_block}
    n_correct  = 0

    for r in records:
        sp_total[r["gt"]] += 1
        if r["correct"]:
            n_correct += 1
            sp_correct[r["gt"]] += 1

    per_species_acc = {
        sp: (100 * sp_correct[sp] / sp_total[sp]
             if sp_total[sp] > 0 else float("nan"))
        for sp in species_in_block
    }

    n_informative = sum(
        1 for r in records
        if r["idx"] not in review_set
        and r["accepted"]
        and CDF_REDUNDANT <= r["cdf"] <= CDF_OUTLIER
    )
    n_redundant = sum(
        1 for r in records
        if r["idx"] not in review_set
        and r["cdf"] < CDF_REDUNDANT
    )
    
    known_species_after = clf.known_species

    return {
        # ── core metrics (compatible with print_results) ──────────────
        "mean_species_accuracy"    : float(np.nanmean(list(per_species_acc.values()))),
        "global_accuracy"          : 100 * n_correct / N if N > 0 else float("nan"),
        "per_species_accuracy"     : per_species_acc,
        "n_correct_classifications": n_correct,
        "total"                    : N,
        "n_known_species_prev"     : len(known_species_prev),
        "n_known_species_after"    : len(known_species_after),
        # ── pipeline diagnostics ──────────────────────────────────────
        "n_reviewed"               : len(review_queue),
        "review_rate"              : len(review_queue) / N,
        "n_informative_updates"    : n_informative,
        "n_redundant_discarded"    : n_redundant,
        "n_confirmed_total"        : len(confirmed),
        "n_actual_new_species_images" : n_actual_new_species_images,
        "actual_new_species"          : actual_new_species,
        "n_new_species_in_queue"      : n_new_species_in_queue,
        "new_species_recall"          : new_species_recall,
    }


# ══════════════════════════════════════════════════════════════════════
# 3.  PIPELINE RUNNER  (drop-in replacement for run_internal_pipeline)
# ══════════════════════════════════════════════════════════════════════
def run_gallery_pipeline(group_blocks, embeddings, species,
                         subprototype_counts = None,
                         label: str = "Group",
                         update_model: bool = True):
    """
    Drop-in replacement for run_internal_pipeline().

    Block 1:  fit on first 80%, evaluate on remaining 20%, then absorb
              the hold-out into the gallery (same as before).
    Blocks 2+: process_block() → evaluate → update gallery.

    Parameters
    ----------
    subprototype_counts : {species: k} dict.  Defaults to 1 everywhere.
                          Pass {"insect": 2} to split insect only.
    update_model        : if False, gallery is never updated (static baseline).
    """
    print("=" * 65)
    print(f"{label}  GALLERY PIPELINE — {'ADAPTIVE' if update_model else 'STATIC'}")
    print("=" * 65)

    if subprototype_counts is None:
        subprototype_counts = {}

    # ── block 1: initialise ──────────────────────────────────────────
    b0_idx  = group_blocks[0]
    n_init  = max(1, int(INIT_TRAIN_RATIO * len(b0_idx)))
    init_idx = b0_idx[:n_init]
    hold_idx = b0_idx[n_init:]

    clf = GalleryNCMClassifier()
    clf.fit(embeddings[init_idx], species[init_idx],
            subprototype_counts=subprototype_counts)

    # evaluate on hold-out (same as your existing baseline row)
    baseline = process_block(
        clf,
        embeddings[hold_idx],
        species[hold_idx],
    ) if update_model else _evaluate_only(clf, embeddings[hold_idx], species[hold_idx])

    results = [{"block": 1, **baseline}]
    _print_block_summary(1, baseline)

    # ── blocks 2–N ───────────────────────────────────────────────────
    for blk_num, blk_idx in enumerate(group_blocks[1:], start=2):
        blk_idx  = np.asarray(blk_idx)
        blk_embs = embeddings[blk_idx]
        blk_lbls = species[blk_idx]

        if update_model:
            metrics = process_block(clf, blk_embs, blk_lbls)
        else:
            metrics = _evaluate_only(clf, blk_embs, blk_lbls)

        results.append({"block": blk_num, **metrics})
        _print_block_summary(blk_num, metrics)

    return results


# ══════════════════════════════════════════════════════════════════════
# 4.  HELPERS
# ══════════════════════════════════════════════════════════════════════
def _evaluate_only(clf: GalleryNCMClassifier,
                   blk_embs: np.ndarray,
                   blk_lbls: np.ndarray) -> dict:
    """Pure evaluation without any gallery update (static baseline)."""
    species_in_block = sorted(set(blk_lbls))
    sp_correct = {sp: 0 for sp in species_in_block}
    sp_total   = {sp: 0 for sp in species_in_block}
    n_correct  = 0
    for emb, lbl in zip(blk_embs, blk_lbls):
        pred = clf.predict(emb)
        sp_total[lbl] += 1
        if pred == lbl:
            n_correct += 1
            sp_correct[lbl] += 1
    N = len(blk_lbls)
    per_species_acc = {
        sp: (100 * sp_correct[sp] / sp_total[sp]
             if sp_total[sp] > 0 else float("nan"))
        for sp in species_in_block
    }
    return {
        "mean_species_accuracy"    : float(np.nanmean(list(per_species_acc.values()))),
        "global_accuracy"          : 100 * n_correct / N if N > 0 else float("nan"),
        "per_species_accuracy"     : per_species_acc,
        "n_correct_classifications": n_correct,
        "total"                    : N,
        "n_known_species_prev"     : len(clf.known_species),
        "n_known_species_after"    : len(clf.known_species),
        "n_reviewed"               : 0,
        "review_rate"              : 0.0,
        "n_informative_updates"    : 0,
        "n_redundant_discarded"    : 0,
        "n_confirmed_total"        : 0,
        "n_actual_new_species_images" : 0,
        "actual_new_species"          : set(),
        "n_new_species_in_queue"      : 0,
        "new_species_recall"          : float("nan"),
    }


def _print_block_summary(blk_num: int, m: dict):
    print(f"\n  Block {blk_num}")
    print(f"    Mean species accuracy : {m['mean_species_accuracy']:.1f}%")
    print(f"    Global accuracy       : {m['global_accuracy']:.1f}%")
    print(f"    Total samples         : {m['total']:,}")
    print(f"    Known species before update        : {m['n_known_species_prev']}")
    print(f"    Known species after update         : {m['n_known_species_after']}")
    
    if m.get("n_reviewed", 0) > 0 or m.get("review_rate", 0) > 0:
        print(f"    Sent for review       : {m['n_reviewed']:,}  "
              f"({m['review_rate']:.1%})")
        print(f"    Informative updates   : {m['n_informative_updates']:,}")
        print(f"    Redundant discarded   : {m['n_redundant_discarded']:,}")
        print(f"    Actual new species images   : {m['n_actual_new_species_images']:,}  "
              f"{m['actual_new_species']}")
        print(f"    New species in review queue : {m['n_new_species_in_queue']:,}")
        recall_str = f"{m['new_species_recall']:.1%}" \
                     if not np.isnan(m['new_species_recall']) else "n/a"
        print(f"    New species recall          : {recall_str}")


def print_results(results):
    """Compatible with your existing print_results signature."""
    for row in results:
        print(f"\n{'='*50}")
        print(f"Block {row['block']}")
        print(f"{'='*50}")
        print(f"  Mean Species Accuracy : {row['mean_species_accuracy']:.1f}%")
        print(f"  Global Accuracy       : {row['global_accuracy']:.1f}%")
        print(f"  Known Species         : {row['n_known_species']}")
        print(f"  Total Samples         : {row['total']:,}")
        print(f"  Correct               : {row['n_correct_classifications']:,}")
        if row.get("n_reviewed", 0) > 0:
            print(f"  Reviewed              : {row['n_reviewed']:,}  "
                  f"({row['review_rate']:.1%})")
            print(f"  Informative updates   : {row['n_informative_updates']:,}")
            print(f"  Redundant discarded   : {row['n_redundant_discarded']:,}")


In [13]:
# ══════════════════════════════════════════════════════════════════════
# 5.  ENTRY POINT  (mirrors your existing run calls)
# ══════════════════════════════════════════════════════════════════════

SP_COUNTS = {"insect": 2}

city_ad   = run_gallery_pipeline(g1_blocks, embeddings, species,
                                 subprototype_counts=SP_COUNTS,
                                 label="City",  update_model=True)

city_st   = run_gallery_pipeline(g1_blocks, embeddings, species,
                                 subprototype_counts=SP_COUNTS,
                                 label="City",  update_model=False)

rural_ad  = run_gallery_pipeline(g2_blocks, embeddings, species,
                                 subprototype_counts=SP_COUNTS,
                                 label="Rural", update_model=True)

rural_st  = run_gallery_pipeline(g2_blocks, embeddings, species,
                                 subprototype_counts=SP_COUNTS,
                                 label="Rural", update_model=False)

City  GALLERY PIPELINE — ADAPTIVE

  Block 1
    Mean species accuracy : 84.7%
    Global accuracy       : 78.5%
    Total samples         : 2,329
    Known species before update        : 15
    Known species after update         : 15
    Sent for review       : 631  (27.1%)
    Informative updates   : 1,405
    Redundant discarded   : 293
    Actual new species images   : 0  set()
    New species in review queue : 0
    New species recall          : n/a

  Block 2
    Mean species accuracy : 85.2%
    Global accuracy       : 78.8%
    Total samples         : 11,339
    Known species before update        : 15
    Known species after update         : 15
    Sent for review       : 2,316  (20.4%)
    Informative updates   : 6,513
    Redundant discarded   : 2,510
    Actual new species images   : 0  set()
    New species in review queue : 0
    New species recall          : n/a

  Block 3
    Mean species accuracy : 83.3%
    Global accuracy       : 79.5%
    Total samples         : 11,4